In [ ]:
#### Celeba-hq or AFHQ-v2 ####
import os
import io
import base64
from pathlib import Path
import pandas as pd
from PIL import Image
import numpy as np
from tqdm import tqdm

#### Celeba-hq ####
# dataset_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/celeba-hq/data"
# target_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/celeba-hq/image_data"

#### AFHQ-v2 ####
dataset_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/afhqv2/data"
target_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/afhqv2/image_dir"

os.makedirs(target_image_dir, exist_ok=True)

parquet_files = sorted([f for f in Path(dataset_dir).glob("*.parquet") if f.name.startswith("train")])
print(f"找到 {len(parquet_files)} 个以 train 开头的 parquet 文件")

image_counter = 0
for parquet_file in parquet_files[0:1]:
    print(f"\n处理文件: {parquet_file.name}")
    
    df = pd.read_parquet(parquet_file)
    
    if 'image' not in df.columns:
        print(f"警告: {parquet_file.name} 中没有 'image' 列")
        continue
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"提取 {parquet_file.stem}"):
        image_data = row['image']
        
        if isinstance(image_data, Image.Image):
            img = image_data
        elif isinstance(image_data, dict):
            if 'bytes' in image_data:
                img = Image.open(io.BytesIO(image_data['bytes']))
            elif 'path' in image_data:
                img = Image.open(image_data['path'])
            else:
                img = Image.fromarray(np.array(image_data))
        elif isinstance(image_data, (bytes, bytearray)):
            img = Image.open(io.BytesIO(image_data))
        elif isinstance(image_data, np.ndarray):
            if image_data.dtype != np.uint8:
                if image_data.max() <= 1.0:
                    image_data = (image_data * 255).astype(np.uint8)
                else:
                    image_data = image_data.astype(np.uint8)
            img = Image.fromarray(image_data)
        elif isinstance(image_data, str):
            if os.path.exists(image_data):
                img = Image.open(image_data)
            else:
                try:
                    img_bytes = base64.b64decode(image_data)
                    img = Image.open(io.BytesIO(img_bytes))
                except:
                    print(f"无法解析图像数据 (索引 {idx})")
                    continue
        else:
            print(f"未知的图像数据格式 (索引 {idx}): {type(image_data)}")
            continue
        
        # 确保是 RGB 模式（如果不是，转换为 RGB）
        if img.mode != 'RGB':
            img = img.convert('RGB')
        
        # 保存图像
        image_filename = f"{parquet_file.stem}_{image_counter:06d}.png"
        image_path = os.path.join(target_image_dir, image_filename)
        img.save(image_path, 'PNG')
        
        image_counter += 1

print(f"\n完成! 共提取 {image_counter} 张图像到 {target_image_dir}")



In [ ]:
#### Extract Pick-a-Pic-v2 ####
import os
import pandas as pd

pick_a_pic_v2_test_prompt_path = "/data3/chenweiyan/notebook/code/data/test_unique-00000-of-00001-ecef20a3469ae6e6.parquet"
df = pd.read_parquet(pick_a_pic_v2_test_prompt_path)
write_txt_path = "/data3/chenweiyan/notebook/fine-tune-diffusion/spo_gitee/DiffusionNFT/dataset/pick_a_pic_v2/test.txt"

# 查看 DataFrame 的基本信息和列名
print("DataFrame 形状:", df.shape)
print("\n列名:", df.columns.tolist())
print("\n前几行数据:")
print(df.head())

captions_series = df['caption']
print(f"\n提取到 {len(captions_series)} 条 caption")

captions_list = df['caption'].tolist()
print(f"\n前5条 caption:")
for i, caption in enumerate(captions_list[:5]):
    print(f"{i+1}. {caption}")

# 将 caption 写入文件
os.makedirs(os.path.dirname(write_txt_path), exist_ok=True)
with open(write_txt_path, 'w', encoding='utf-8') as f:
    for caption in captions_list:
        f.write(str(caption) + '\n')
print(f"\n已将 {len(captions_list)} 条 caption 写入到: {write_txt_path}")

